# Analisis Eksploratif dan Pembersihan Data: Ames Housing Dataset

Dokumentasi proyek pembersihan data mentah (*raw data preprocessing*) sebelum tahap pemodelan regresi.

---

## 1. Pendahuluan & Ringkasan Tahapan Kerja

Kualitas model pembelajaran mesin sangat bergantung pada integritas data yang digunakan (*garbage in, garbage out*). Dataset yang dianalisis mencakup data transaksi properti di Ames, Iowa. Fokus pembersihan meliputi:
1. **Pemeriksaan Konsistensi Kategori** pada variabel non-numerik.
2. **Uji Asimetri Distribusi (Skewness)** dan normalisasi menggunakan transformasi logaritmik.
3. **Deteksi dan Eliminasi Entri Duplikat** pada indeks identitas transaksi.
4. **Penanganan Nilai Hilang (Imputasi)** dengan pendekatan ukuran pemusatan data (mean & median).
5. **Penskalaan Fitur Numerik** menggunakan Standardisasi (*Z-Score*) dan Normalisasi (*Min-Max*).
6. **Skrining Nilai Ekstrem (Outlier)** melalui visualisasi multivariat dan uji ambang batas statistik.

## 2. Inisialisasi Dependensi

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Pengaturan tema visualisasi
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (9, 5)

## 3. Pemuatan dan Penelaahan Awal Data

In [ ]:
DATA_SOURCE_URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/Ames_Housing_Data1.tsv'
df_property = pd.read_csv(DATA_SOURCE_URL, sep='\t')

print(f'Dimensi dataset awal: {df_property.shape[0]} baris, {df_property.shape[1]} kolom')
df_property.head(3)

In [ ]:
# Tinjauan statistik ringkas untuk variabel target: SalePrice
price_stats = df_property['SalePrice'].describe()
display(price_stats.to_frame())

### Tugas 1: Telaah Distribusi Kategori Variabel `Sale Condition`

In [ ]:
# Menganalisis sebaran frekuensi absolut dan proporsi relatif kondisi penjualan
cond_counts = df_property['Sale Condition'].value_counts()
cond_percentages = df_property['Sale Condition'].value_counts(normalize=True).mul(100).round(2)

summary_sale_condition = pd.DataFrame({
    'Frekuensi': cond_counts,
    'Proporsi (%)': cond_percentages
})
summary_sale_condition

## 4. Analisis Asosiasi Fitur (Korelasi Linier terhadap Target)

In [ ]:
# Ekstraksi kolom numerik dan hitung koefisien korelasi Pearson
num_subset = df_property.select_dtypes(include=['float64', 'int64'])
correlations = num_subset.corr()['SalePrice'].drop('SalePrice')

# Menyaring korelasi kuat (|r| > 0.5)
strong_corrs = correlations[correlations.abs() > 0.5].sort_values(ascending=False)
print(f'Terdapat {len(strong_corrs)} prediktor dengan korelasi kuat (|r| > 0.5):')
display(strong_corrs.to_frame(name='Korelasi Pearson'))

## 5. Pemeriksaan Skewness dan Transformasi Logaritmik

Banyak algoritma regresi linier mensyaratkan asumsi normalitas pada residual. Jika suatu fitur memiliki kemiringan ke kanan (*right-skewed*), transformasi logaritmik dapat menormalkan distribusinya.

In [ ]:
skew_original = df_property['SalePrice'].skew()
log_saleprice = np.log(df_property['SalePrice'])
skew_transformed = log_saleprice.skew()

fig, axs = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df_property['SalePrice'], kde=True, ax=axs[0], color='navy')
axs[0].set_title(f'SalePrice Asli (Skewness: {skew_original:.3f})')

sns.histplot(log_saleprice, kde=True, ax=axs[1], color='teal')
axs[1].set_title(f'SalePrice Log-Transformed (Skewness: {skew_transformed:.3f})')
plt.tight_layout()
plt.show()

### Tugas 2: Pemeriksaan dan Transformasi Log pada Fitur `Lot Area`

In [ ]:
skew_lot_orig = df_property['Lot Area'].skew()
lot_area_log = np.log(df_property['Lot Area'])
skew_lot_log = lot_area_log.skew()

print(f'Skewness Lot Area sebelum transformasi: {skew_lot_orig:.3f}')
print(f'Skewness Lot Area setelah transformasi : {skew_lot_log:.3f}')

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df_property['Lot Area'], kde=True, ax=ax[0], color='darkred')
ax[0].set_title('Distribusi Lot Area (Asli)')
sns.histplot(lot_area_log, kde=True, ax=ax[1], color='forestgreen')
ax[1].set_title('Distribusi Lot Area (Setelah Log-Transform)')
plt.tight_layout()
plt.show()

## 6. Identifikasi dan Penanganan Baris Duplikat

In [ ]:
# Menemukan duplikasi record berdasarkan identifier properti unik (PID)
duplicate_mask = df_property.duplicated(subset=['PID'], keep=False)
duplicates_found = df_property[duplicate_mask]
print(f'Jumlah baris yang terindikasi duplikat pada PID: {len(duplicates_found)}')
display(duplicates_found[['Order', 'PID', 'Lot Area', 'SalePrice']])

In [ ]:
# Menghapus entri duplikat dan mempertahankan kejadian pertama
df_property = df_property.drop_duplicates(subset=['PID'], keep='first')
print(f'Ukuran dataset setelah drop_duplicates pada PID: {df_property.shape}')

### Tugas 3: Uji Penghapusan Duplikat Berdasarkan Subset Kolom `Order`

In [ ]:
# Mengeksekusi drop_duplicates dengan parameter subset=['Order']
df_dedup_order = df_property.drop_duplicates(subset=['Order'])
print(f'Jumlah record setelah verifikasi subset Order: {df_dedup_order.shape[0]}')

## 7. Penanganan Nilai Hilang (*Missing Values*)

Memetakan proporsi nilai null di setiap variabel untuk menentukan strategi penanganan yang tepat (dropping vs imputasi).

In [ ]:
null_series = df_property.isnull().sum()
top_missing = null_series[null_series > 0].sort_values(ascending=False).head(20)

top_missing.plot(kind='bar', color='royalblue', edgecolor='black')
plt.title('20 Kolom dengan Missing Values Terbanyak')
plt.ylabel('Frekuensi Nilai Hilang')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Imputasi fitur 'Lot Frontage' menggunakan nilai median agar tahan terhadap outlier
val_median_frontage = df_property['Lot Frontage'].median()
df_property['Lot Frontage'] = df_property['Lot Frontage'].fillna(val_median_frontage)
print(f'Median Lot Frontage: {val_median_frontage:.2f} | Sisa null: {df_property["Lot Frontage"].isnull().sum()}')

### Tugas 4: Imputasi Nilai Hilang Fitur `Mas Vnr Area` dengan Mean

In [ ]:
# Menghitung rata-rata dan mengisi nilai null pada 'Mas Vnr Area'
val_mean_masvnr = df_property['Mas Vnr Area'].mean()
df_property['Mas Vnr Area'] = df_property['Mas Vnr Area'].fillna(val_mean_masvnr)
print(f'Nilai mean Mas Vnr Area: {val_mean_masvnr:.3f}')
print(f'Sisa missing value pada Mas Vnr Area: {df_property["Mas Vnr Area"].isnull().sum()}')

## 8. Penskalaan Fitur (*Feature Scaling*)

Membandingkan hasil transformasi Standardisasi ($Z = \frac{X - \mu}{\sigma}$) dan Normalisasi Min-Max ($X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$).

In [ ]:
# Menyiapkan subset fitur numerik bebas nilai null sebelum penskalaan
num_clean = num_subset.drop_duplicates(subset=['PID'], keep='first').fillna(num_subset.median())

# Normalisasi Min-Max
scaler_minmax = MinMaxScaler()
arr_normalized = scaler_minmax.fit_transform(num_clean)
print('Dimensi array ter-normalisasi:', arr_normalized.shape)

# Standardisasi Z-score
scaler_std = StandardScaler()
arr_standardized = scaler_std.fit_transform(num_clean)
print('Dimensi array ter-standarisasi:', arr_standardized.shape)

### Tugas 5: Standardisasi Mandiri pada Variabel Target `SalePrice`

In [ ]:
# Menerapkan StandardScaler khusus pada kolom SalePrice (input 2-dimensi)
sprice_std_scaler = StandardScaler()
scaled_target = sprice_std_scaler.fit_transform(df_property[['SalePrice']])

print(f'Mean hasil standardisasi: {scaled_target.mean():.4f} (mendekati 0)')
print(f'Standar deviasi hasil standardisasi: {scaled_target.std():.4f} (mendekati 1)')
scaled_target[:5]

## 9. Deteksi dan Penghapusan Nilai Ekstrem (*Outliers*)

In [ ]:
# Analisis bivariat: Hubungan Gr Liv Area terhadap SalePrice
plt.figure(figsize=(8, 5))
plt.scatter(df_property['Gr Liv Area'], df_property['SalePrice'], alpha=0.5, c='darkslategray')
plt.xlabel('Ground Living Area (Gr Liv Area)')
plt.ylabel('Sale Price')
plt.title('Scatter Plot: Deteksi Anomali Luas Bangunan vs Harga')
plt.show()

In [ ]:
# Mengeliminasi 2 observasi ekstrim (> 5000 sq ft) yang menyimpang dari tren pasar
extreme_liv_idx = df_property.sort_values(by='Gr Liv Area', ascending=False).index[:2]
df_property = df_property.drop(extreme_liv_idx)
print(f'Record setelah eliminasi outlier Gr Liv Area: {df_property.shape[0]}')

### Tugas 6: Identifikasi dan Eliminasi Outlier pada Fitur `Lot Area`

In [ ]:
# Visualisasi Boxplot dan evaluasi Z-Score statistik
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(x=df_property['Lot Area'], ax=ax[0], color='lightcoral')
ax[0].set_title('Boxplot Sebaran Lot Area')

ax[1].scatter(df_property['Lot Area'], df_property['SalePrice'], alpha=0.5, color='purple')
ax[1].set_xlabel('Lot Area')
ax[1].set_ylabel('SalePrice')
ax[1].set_title('Bivariat: Lot Area vs SalePrice')
plt.tight_layout()
plt.show()

# Hitung skor standar Z
df_property['Lot_Area_Zscore'] = stats.zscore(df_property['Lot Area'])
display(df_property[['Lot Area', 'Lot_Area_Zscore']].describe().round(3))

# Menghapus 1 nilai ekstrem terbesar yang memiliki Z-score jauh melebihi threshold 3
lot_outlier_idx = df_property.sort_values(by='Lot Area', ascending=False).index[:1]
df_property_cleaned = df_property.drop(lot_outlier_idx)
print(f'Total baris akhir setelah dataset bersih: {df_property_cleaned.shape[0]}')

## 10. Kesimpulan Pembersihan Data

* Target `SalePrice` dan prediktor `Lot Area` berhasil ditransformasikan secara logaritmik untuk mereduksi kemiringan kurva (*skewness*).
* Integritas entri data terjaga dengan menghapus duplikasi transaksi unik.
* Imputasi berhasil menghilangkan nilai kosong tanpa mendistorsi varians data.
* Outlier ekstrem pada `Gr Liv Area` dan `Lot Area` yang berpotensi merusak estimasi regresi telah dibersihkan dari dataset.